# 3.2 Classification et régression

🖥️ [**Diapositives du cours : séance 12 (lun. 26 oct.)**](https://geo-smart.github.io/mlgeo-book/slides/2026/lec12_supervision_concepts.html)

**Objectifs d'apprentissage.** À la fin de ce notebook, vous saurez :

1. entraîner un classifieur sur les étiquettes d'analystes d'un vrai catalogue sismique, et lire sa matrice de confusion et le rappel de chaque classe par rapport à un modèle de référence (*baseline*) qui répond toujours la classe majoritaire ;
2. choisir un hyperparamètre sur un ensemble de validation tiré du domaine d'entraînement, puis évaluer une seule fois le modèle retenu sur un ensemble de test pris hors de ce domaine (une période plus récente, ou une autre composition de roche) ;
3. repérer une variable raccourci, qui sépare les classes dans le catalogue pour une raison qui ne vaudra pas pour tous les événements, et mesurer ce que le modèle perd ou gagne sans elle ;
4. entraîner un modèle de régression pour une grandeur physique, donner son erreur en unités physiques, et montrer que l'erreur augmente quand les échantillons de test sortent de la gamme d'entraînement.

**La classification** prédit une catégorie : séisme ou explosion (tir de carrière ou de mine), type de roche, crue ou absence de crue. **La régression** prédit une grandeur (masse volumique en g/cm³, débit en m³/s, magnitude). La démarche est la même pour les deux : définir la cible et les variables, partitionner les données, ajuster une référence triviale, ajuster des modèles, choisir entre eux sur les données de validation, tester une seule fois. Seule la métrique change : effectifs et rappels par classe pour la classification, erreurs dans les unités de la cible pour la régression.

**Pour l'exécuter.** Depuis la racine du dépôt, lancez `pixi run jupyter lab`. Exécutez une fois la cellule de configuration ci-dessous. La section 1 télécharge deux tables du catalogue de séismes de l'USGS (environ 3 MB, quelques secondes) et les garde dans un cache local ; sans réseau, le notebook le signale et utilise à la place un catalogue synthétique. Chaque cellule **À vous** est suivie d'une vérification qui affiche `correct`, un indice ou `not answered yet`. Si votre réponse n'est pas encore correcte, la vérification met en place le résultat de référence pour que la suite du notebook tourne ; votre propre résultat est conservé avec le suffixe `_yours`. Les sections 1 à 5 tiennent en 45 minutes.

## Organisation de ce notebook

La première cellule de code importe tous les paquets utilisés dans le notebook. Placer les imports en tête donne un seul endroit où voir ce dont l'analyse dépend (tout est fourni par l'environnement du cours) et fait échouer un paquet manquant dès le début plutôt qu'à mi-parcours.

Le code utilisé plusieurs fois, ou qui encombrerait une cellule d'analyse (télécharger un catalogue, mettre en forme une figure, disposer un tableau), est écrit sous forme de **fonction** nommée et défini juste avant sa première utilisation. Une fonction au nom clair, avec une ligne de description, permet à chaque cellule d'analyse de montrer les étapes de l'analyse ; ouvrez une cellule repliée pour voir comment une figure ou un tableau est construit.


In [ ]:
# Data handling
import warnings

import numpy as np
import pandas as pd
import pooch                      # downloads a file once and keeps a local copy

# Plotting
import matplotlib.pyplot as plt

# Machine learning (scikit-learn)
from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (ConfusionMatrixDisplay, balanced_accuracy_score, confusion_matrix,
                             mean_absolute_error, r2_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Course data generators (the Session 5 rock table)
import mlgeo_synth


Les valeurs ci-dessous décrivent la zone d'étude et la source des données. Elles servent à plusieurs endroits, donc elles sont définies une seule fois : changer ici la région ou le fuseau horaire les change partout.


In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)   # hide library deprecation notices
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
CLASSES = ["earthquake", "explosion"]
COLORS = {"earthquake": "C0", "explosion": "C3", "basalt": "C3", "andesite": "C2", "granite": "C0"}
TZ = "America/Los_Angeles"
BOX = dict(minlatitude=42, maxlatitude=49.5, minlongitude=-125.5, maxlongitude=-116.5)
COMCAT = "https://earthquake.usgs.gov/fdsnws/event/1/query"
CACHE = pooch.os_cache("mlgeo")
SOURCE = {}


La cellule suivante, repliée, définit les vérifications des cellules **À vous**. Chaque vérification compare votre réponse à une référence, affiche *correct* ou un indice et, si votre cellule est inachevée, utilise la référence pour que la suite du notebook s'exécute. Vous n'avez pas besoin de la lire.


In [ ]:
# Checks for the Your-turn cells: each compares your answer with a reference and prints a hint.
TURNS = {"H": ["hour"], "W": ["work_share"], "B": ["acc_base"], "K": ["val_scores", "best_k"],
         "T": ["knn", "y_pred_test", "recall_test"], "N": ["y_pred_nohour"], "R1": ["base_r", "lin", "val_mae"],
         "R2": ["ood_mae", "frac_physical"]}
HINTS = {
    "H": "hour = cat['time'].dt.tz_convert('America/Los_Angeles').dt.hour",
    "W": "work_share = cat.groupby('label')['hour'].apply(lambda h: h.between(8, 16).mean()); "
         "plt.hist(cat.loc[cat['label'] == lab, 'hour'], bins=np.arange(25), density=True, alpha=0.5) for each label.",
    "B": "baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train); acc_base = baseline.score(X_val, y_val)",
    "K": "for k in K_LIST: fit make_knn(k) on (X_train, y_train) and store balanced_accuracy_score(y_val, model.predict(X_val)) "
         "in val_scores[k]; best_k = max(val_scores, key=val_scores.get)",
    "T": "knn = make_knn(best_k).fit(X_train, y_train); y_pred_test = knn.predict(X_test); "
         "recall_test = recall_score(y_test, y_pred_test, labels=CLASSES, average=None)",
    "N": "fit make_knn(best_k) on X_train[NOHOUR] and predict X_test[NOHOUR], with NOHOUR the feature list without 'hour'.",
    "R1": "base_r = DummyRegressor(strategy='mean').fit(Xr_train, yr_train); lin = LinearRegression().fit(Xr_train, yr_train); "
          "val_mae = {'mean': mean_absolute_error(yr_val, base_r.predict(Xr_val)), 'linear': ...}",
    "R2": "ood_mae = {name: mean_absolute_error(yr_ood, m.predict(Xr_ood)) for name, m in models.items()}; "
          "p = models[best_name].predict(Xr_ood); frac_physical = ((p >= 2.5) & (p <= 3.3)).mean()",
}


def _ref(key, ns):
    r = {}
    if key == "H":
        r["hour"] = ns["cat"]["time"].dt.tz_convert(TZ).dt.hour
    elif key == "W":
        r["work_share"] = ns["cat"].groupby("label")["hour"].apply(lambda h: h.between(8, 16).mean())
    elif key == "B":
        r["acc_base"] = DummyClassifier(strategy="most_frequent").fit(ns["X_train"], ns["y_train"]).score(ns["X_val"], ns["y_val"])
    elif key == "K":
        r["val_scores"] = {k: balanced_accuracy_score(ns["y_val"], make_knn(k).fit(ns["X_train"], ns["y_train"]).predict(ns["X_val"]))
                           for k in ns["K_LIST"]}
        r["best_k"] = max(r["val_scores"], key=r["val_scores"].get)
    elif key == "T":
        r["knn"] = make_knn(ns["best_k"]).fit(ns["X_train"], ns["y_train"])
        r["y_pred_test"] = r["knn"].predict(ns["X_test"])
        r["recall_test"] = recall_score(ns["y_test"], r["y_pred_test"], labels=CLASSES, average=None)
    elif key == "N":
        f = [c for c in ns["FEATURES"] if c != "hour"]
        r["y_pred_nohour"] = make_knn(ns["best_k"]).fit(ns["X_train"][f], ns["y_train"]).predict(ns["X_test"][f])
    elif key == "R1":
        r["base_r"] = DummyRegressor(strategy="mean").fit(ns["Xr_train"], ns["yr_train"])
        r["lin"] = LinearRegression().fit(ns["Xr_train"], ns["yr_train"])
        r["val_mae"] = {n: mean_absolute_error(ns["yr_val"], r[m].predict(ns["Xr_val"]))
                        for n, m in [("mean", "base_r"), ("linear", "lin")]}
    elif key == "R2":
        r["ood_mae"] = {n: mean_absolute_error(ns["yr_ood"], m.predict(ns["Xr_ood"])) for n, m in ns["models"].items()}
        p = ns["models"][ns["best_name"]].predict(ns["Xr_ood"])
        r["frac_physical"] = float(((p >= 2.5) & (p <= 3.3)).mean())
    return r


def _close(a, b, rtol=1e-3):
    if isinstance(b, dict):
        return isinstance(a, dict) and set(a) == set(b) and all(_close(a[k], b[k], rtol) for k in b)
    try:
        a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    except (TypeError, ValueError):
        return False
    return a.shape == b.shape and bool(np.allclose(a, b, rtol=rtol, atol=1e-9))


def _verdict(key, v, r):
    hint = "not yet. Hint: " + HINTS[key]
    if key == "W":
        w = v["work_share"]
        try:
            w = pd.Series(w).reindex(CLASSES)
        except Exception:
            return hint
        return "correct" if _close(w.to_numpy(), r["work_share"].reindex(CLASSES).to_numpy()) else hint
    if key in ("T", "N"):
        n = TURNS[key][1] if key == "T" else TURNS[key][0]
        if key == "T" and not _close(v["recall_test"], r["recall_test"]):
            return hint
        ok = v[n] is not None and len(v[n]) == len(r[n]) and np.mean(np.asarray(v[n]) == r[n]) > 0.999
        return "correct" if ok else hint
    if key == "K":
        ok = _close(v["val_scores"], r["val_scores"]) and v["best_k"] == r["best_k"]
        return "correct" if ok else hint
    if key == "R1":
        ok = _close(v["val_mae"], r["val_mae"]) and v["lin"] is not None and v["base_r"] is not None
        return "correct" if ok else hint
    ok = all(_close(v[n], r[n]) for n in TURNS[key])
    return "correct" if ok else hint


def your_turn(key, ns):
    r = _ref(key, ns)
    v = {n: ns.get(n) for n in TURNS[key]}
    verdict = "not answered yet" if all(x is None for x in v.values()) else _verdict(key, v, r)
    print(f"[{key}] {verdict}")
    if verdict == "correct":
        return
    kept = []
    for n in TURNS[key]:
        if ns.get(n) is not None:
            ns[n + "_yours"] = ns[n]
            kept.append(n + "_yours")
        ns[n] = r[n]
    note = f"; yours kept as {', '.join(kept)}" if kept else ""
    print(f"[{', '.join(TURNS[key])}] using the reference result so the next cells run{note}.")
    if key == "W":
        plot_hours(ns["cat"])


## 1. Séisme ou explosion ? Le catalogue et ses étiquettes

Le Pacific Northwest Seismic Network (PNSN), le réseau sismique du nord-ouest des États-Unis, localise quelques milliers d'événements par an dans l'État de Washington et en Oregon. Tous ne sont pas des séismes : les carrières, les mines à ciel ouvert et les chantiers de construction font des tirs à l'explosif que le réseau enregistre aussi. Un analyste du PNSN examine chaque événement et lui attribue un type ; les tirs sont étiquetés `explosion`. Ces étiquettes d'analystes sont la cible de notre classifieur. Elles sont publiées dans le USGS Comprehensive Catalog (ComCat), que nous interrogeons pour 2015 à 2024 dans un rectangle de 42 à 49,5°N et de 125,5 à 116,5°W : les séismes de magnitude 1 et plus, et toutes les explosions.

Les tirs ont lieu aux heures de travail, en surface, à côté d'une fosse d'extraction. Les séismes se produisent à toute heure, à des profondeurs de quelques kilomètres à plusieurs dizaines de kilomètres. Wiemer et Baer (2000, *Bulletin of the Seismological Society of America* 90, 525-530, [doi:10.1785/0119990104](https://doi.org/10.1785/0119990104)) ont utilisé le premier fait pour trouver les régions d'un catalogue contaminées par des tirs : là où le rapport entre événements de jour et événements de nuit est élevé, le catalogue contient des tirs. Nous posons la même question, événement par événement.

La fonction ci-dessous télécharge le catalogue depuis ComCat une seule fois et en garde une copie locale : le notebook s'exécute rapidement la deuxième fois et fonctionne un autre jour avec les mêmes données. Si ComCat est inaccessible, elle construit à la place un petit catalogue synthétique (cellule repliée) et le signale.


In [ ]:
# Fallback used only when ComCat cannot be reached: a small synthetic catalog with the same columns.
def synthetic_catalog(seed=42):
    """Small stand-in catalog: earthquakes at any hour and depth, blasts at 12 'quarries' during working hours."""
    rng = np.random.default_rng(seed)
    n_eq, n_ex = 3000, 1600
    t0, t1 = pd.Timestamp("2015-01-01", tz="UTC").value, pd.Timestamp("2025-01-01", tz="UTC").value
    eq = pd.DataFrame({"time": pd.to_datetime(rng.uniform(t0, t1, n_eq).astype("int64"), utc=True),
                       "latitude": rng.uniform(42, 49.5, n_eq), "longitude": rng.uniform(-125.5, -116.5, n_eq),
                       "depth": rng.gamma(2.0, 7.0, n_eq), "mag": 1 + rng.exponential(0.45, n_eq),
                       "label": "earthquake"})
    sites = np.column_stack([rng.uniform(43, 49.3, 12), rng.uniform(-123.5, -117, 12)])
    k = rng.integers(0, 12, n_ex)
    day = pd.to_datetime(rng.uniform(t0, t1, n_ex).astype("int64"), utc=True).tz_convert(TZ).normalize()
    local = day + pd.to_timedelta(np.clip(rng.normal(12.5, 2.0, n_ex), 0, 23.9), unit="h")
    ex = pd.DataFrame({"time": local.tz_convert("UTC"), "latitude": sites[k, 0] + rng.normal(0, 0.02, n_ex),
                       "longitude": sites[k, 1] + rng.normal(0, 0.02, n_ex), "depth": rng.normal(-0.5, 0.3, n_ex),
                       "mag": 1 + rng.exponential(0.5, n_ex), "label": "explosion"})
    return pd.concat([eq, ex], ignore_index=True).sort_values("time", ignore_index=True)


In [ ]:
def _comcat_url(eventtype, minmag=None):
    q = dict(format="csv", starttime="2015-01-01", endtime="2025-01-01", eventtype=eventtype, orderby="time-asc", **BOX)
    if minmag is not None:
        q["minmagnitude"] = minmag
    return COMCAT + "?" + "&".join(f"{k}={v}" for k, v in q.items())


def load_catalog():
    """PNSN earthquakes (M >= 1) and explosions from USGS ComCat, 2015-2024, cached; synthetic catalog if offline."""
    try:
        frames = []
        for etype, minmag in [("earthquake", 1), ("explosion", None)]:
            path = pooch.retrieve(_comcat_url(etype, minmag), known_hash=None, path=CACHE, progressbar=False,
                                  fname=f"comcat_pnw_2015-2024_{etype}.csv",
                                  downloader=pooch.HTTPDownloader(timeout=60))
            frames.append(pd.read_csv(path))
        raw = pd.concat(frames, ignore_index=True)
        raw["time"] = pd.to_datetime(raw["time"], utc=True)
        raw["label"] = raw["type"]
        print(f"USGS ComCat, read from the cache {CACHE}:")
        SOURCE["catalog"] = "ComCat"
        return raw[["time", "latitude", "longitude", "depth", "mag", "magType", "depthError", "net", "label"]]
    except Exception as err:
        print(f"FALLBACK: USGS ComCat could not be reached ({type(err).__name__}: {str(err)[:80]}).\n"
              "Using a small synthetic catalog built in the setup cell. Its numbers are NOT the Pacific Northwest;\n"
              "the text describes the real catalog.")
        SOURCE["catalog"] = "synthetic"
        cat = synthetic_catalog()
        cat["magType"], cat["depthError"], cat["net"] = "ml", np.nan, "synthetic"
        return cat


In [ ]:
cat = load_catalog()
print(cat["label"].value_counts().to_string())
cat.head(3)

**La coupure en magnitude.** Les séismes ont été demandés avec M ≥ 1 et les explosions sans coupure : tout événement sous M 1 dans la table est donc une explosion par construction de la requête, et non par la physique. Un classifieur apprendrait cette règle en quelques secondes. Nous appliquons la même coupure aux deux classes. Trois explosions portent la magnitude -5,0, une valeur de remplissage et non une mesure ; la coupure les retire aussi.

In [ ]:
# Helper: the map of events with depth histograms, and the histogram of local hour, colored by label. Kept as functions so the analysis cells show the analysis, not the plot styling.
def plot_catalog(cat):
    fig, ax = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={"width_ratios": [1, 1]})
    try:
        ne = pd.read_csv("../Chapter2-DataManipulation/snapshots/naturalearth_pnw_map.csv")   # Natural Earth 1:50m
        for (kind, part), g in ne[ne["kind"] != "place"].groupby(["kind", "part"]):
            ax[0].plot(g["lon"], g["lat"], color="0.3" if kind == "coastline" else "0.65", lw=0.7)
        for _, p in ne[ne["kind"] == "place"].iterrows():
            ax[0].plot(p["lon"], p["lat"], "ks", ms=4)
            ax[0].annotate(p["name"], (p["lon"], p["lat"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    except FileNotFoundError:
        pass
    for lab in CLASSES:
        m = cat["label"] == lab
        ax[0].scatter(cat.loc[m, "longitude"], cat.loc[m, "latitude"], s=3, alpha=0.4, color=COLORS[lab],
                      label=f"{lab} ({m.sum()})")
    ax[0].set(xlim=(-125.5, -116.5), ylim=(42, 49.5), xlabel="longitude (°)", ylabel="latitude (°)",
              title=f"Events 2015-2024, M ≥ 1 ({SOURCE['catalog']})")
    ax[0].set_aspect(1 / np.cos(np.radians(46)))
    ax[0].legend(fontsize=8, markerscale=4, loc="lower left")
    for lab in CLASSES:
        d = cat.loc[cat["label"] == lab, "depth"]
        ax[1].hist(d, bins=np.arange(-4, 41, 1), alpha=0.5, density=True, color=COLORS[lab], label=lab)
    ax[1].set(xlabel="depth below sea level (km)", ylabel="density", title="Depth by label (negative = above sea level)")
    ax[1].legend(fontsize=8)
    fig.tight_layout()


def plot_hours(cat):
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for lab in CLASSES:
        ax.hist(cat.loc[cat["label"] == lab, "hour"], bins=np.arange(25), alpha=0.5, density=True,
                color=COLORS[lab], label=lab)
    ax.axvspan(8, 17, color="0.85", zorder=0, label="08:00-17:00")
    ax.set(xlabel="local hour (Pacific time)", ylabel="fraction of events per hour", xticks=range(0, 25, 3))
    ax.legend(fontsize=8)


In [ ]:
cat = cat[cat["mag"] >= 1].reset_index(drop=True)
print(cat["label"].value_counts().to_string())
plot_catalog(cat)

Les explosions se regroupent sur les sites de carrières et de mines. Elles sont les plus denses le long du corridor de l'Interstate 5, des basses terres de Puget (Puget Lowland) à la vallée de la Willamette, où l'on extrait des granulats pour la construction, et dispersées dans l'est de l'État de Washington et de l'Oregon. Les séismes sont répartis plus largement, les plus denses sous le Puget Sound et le long de l'arc des Cascades. L'histogramme des profondeurs sépare presque complètement les classes : environ 98 % des explosions sont à moins de 1 km de profondeur, contre environ 9 % des séismes. Une partie de cette séparation ne relève pas de la sismologie. Environ 98 % des explosions portent la même incertitude de profondeur, 31,61 km, et beaucoup ont des profondeurs identiques (-0,37 ou -0,53 km) : l'analyste a fixé la profondeur en surface une fois l'événement classé comme tir. **La profondeur est en partie une conséquence de l'étiquette.** Gardez cela en tête quand nous lirons les scores.

**À vous : heure locale.** Les temps de ComCat sont en UTC. Écrivez `hour`, l'heure de chaque événement (0 à 23) en heure du Pacifique, heure d'été comprise.

*Indice :* `cat["time"].dt.tz_convert("America/Los_Angeles").dt.hour`.

In [ ]:
# --- your code
hour = None   # Series of integers 0-23, one per event
# --- end
your_turn("H", globals())

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [ ]:
# Worked solution, open after you tried
# Names end in _sol so that running this cell keeps your own results.
hour_sol = cat["time"].dt.tz_convert("America/Los_Angeles").dt.hour

**À vous : heure du jour par étiquette.** La première ligne, fournie, ajoute l'heure au catalogue. Tracez un histogramme de `hour` par étiquette sur les mêmes axes (24 intervalles d'une heure, normalisés avec `density=True`), et écrivez `work_share`, la fraction de chaque classe entre 08:00 et 16:59, heure locale (une Series indexée par étiquette).

*Indice :* `cat.groupby("label")["hour"].apply(lambda h: h.between(8, 16).mean())`.

In [ ]:
cat["hour"] = hour   # given
# --- your code
work_share = None   # Series: label -> fraction of events with 8 <= hour <= 16
# --- end
your_turn("W", globals())

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [ ]:
# Worked solution, open after you tried
# Names end in _sol so that running this cell keeps your own results.
fig, ax = plt.subplots(figsize=(7, 3.5))
for lab in CLASSES:
    ax.hist(cat.loc[cat["label"] == lab, "hour"], bins=np.arange(25), density=True, alpha=0.5, label=lab)
ax.set(xlabel="local hour", ylabel="fraction per hour"); ax.legend()
work_share_sol = cat.groupby("label")["hour"].apply(lambda h: h.between(8, 16).mean())

In [ ]:
weekend = cat["time"].dt.tz_convert(TZ).dt.dayofweek >= 5
print("share between 08:00 and 16:59 local:", work_share.round(3).to_dict(), "  (9 of 24 hours = 0.375)")
print("share on Saturday or Sunday:", weekend.groupby(cat["label"]).mean().round(3).to_dict(), "  (2 of 7 days = 0.286)")

Les séismes suivent l'horloge de façon uniforme : leur part dans les neuf heures de travail est proche de 9/24, et leur part le week-end proche de 2/7. Les tirs tombent presque tous dans la journée de travail, avec un pic en début d'après-midi, et rarement le week-end. L'heure du jour est une variable forte, et c'est une variable sociale : elle décrit l'horaire de travail de la mine, pas la source.

## 2. Partitions : dans le domaine et hors domaine dans le temps

Nous suivons la règle de partition de 2.13 (critère B7). Le **domaine** est 2015 à 2021. L'entraînement et la validation en sont tirés, l'ensemble de validation étant tiré au hasard parmi ces événements (stratifié par étiquette) : la validation teste l'interpolation et sert à choisir l'hyperparamètre. L'ensemble de **test** est 2022 à 2024, une période plus récente. Il mesure si un modèle entraîné sur les années passées fonctionne encore sur les événements que le réseau étiquettera ensuite, quand des mines ouvrent et ferment, que le réseau change et que les analystes changent. Le test ne sert qu'une fois, à la fin.

In [ ]:
FEATURES = ["hour", "depth", "mag", "latitude", "longitude"]
in_domain = cat["time"] < pd.Timestamp("2022-01-01", tz="UTC")
X_train, X_val, y_train, y_val = train_test_split(cat.loc[in_domain, FEATURES], cat.loc[in_domain, "label"],
                                                  test_size=0.2, stratify=cat.loc[in_domain, "label"], random_state=42)
X_test, y_test = cat.loc[~in_domain, FEATURES], cat.loc[~in_domain, "label"]
pd.DataFrame({"train 2015-2021": y_train.value_counts(), "validation 2015-2021": y_val.value_counts(),
              "test 2022-2024": y_test.value_counts()})

**À vous : référence de la classe majoritaire.** Un modèle doit faire mieux que le classifieur qui répond toujours la classe la plus fréquente. Ajustez `DummyClassifier(strategy="most_frequent")` sur l'ensemble d'entraînement et écrivez `acc_base`, son exactitude sur l'ensemble de validation.

*Indice :* `.fit(X_train, y_train)`, puis `.score(X_val, y_val)`.

In [ ]:
# --- your code
acc_base = None   # accuracy of the majority-class baseline on the validation set
# --- end
your_turn("B", globals())

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [ ]:
# Worked solution, open after you tried
# Names end in _sol so that running this cell keeps your own results.
baseline_sol = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
acc_base_sol = baseline_sol.score(X_val, y_val)

In [ ]:
print(f"baseline accuracy on validation: {acc_base:.3f}; recall: earthquake 1.0, explosion 0.0; balanced accuracy 0.5")

In [ ]:
# Helper: A k-nearest-neighbors classifier with feature scaling in front. A function because the same model is built three times: to choose k, to test once, and without the hour.
def make_knn(k):
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))


def plot_k_curve(val_scores, best_k):
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.plot(list(val_scores), list(val_scores.values()), "o-", color="k")
    ax.axvline(best_k, color="C3", ls=":", label=f"best k = {best_k}")
    ax.set(xscale="log", xlabel="number of neighbors k", ylabel="balanced accuracy (validation)")
    ax.legend(fontsize=8)


La référence a raison environ deux fois sur trois, et elle ne trouve jamais un tir. L'exactitude récompense la classe majoritaire ; à partir d'ici, nous choisissons donc les modèles avec l'**exactitude équilibrée**, la moyenne des rappels par classe (rappel d'une classe = fraction de ses événements que le modèle étiquette correctement). La référence y obtient 0,5.

**À vous : choisissez k sur l'ensemble de validation.** Le classifieur est celui des k plus proches voisins sur des variables standardisées (`make_knn(k)`, dans la cellule de configuration, construit le *pipeline* scikit-learn `StandardScaler` puis `KNeighborsClassifier`). Pour chaque `k` de `K_LIST`, ajustez sur l'ensemble d'entraînement et rangez l'exactitude équilibrée de validation dans le dictionnaire `val_scores` ; puis donnez à `best_k` la valeur de `k` qui a le meilleur score.

*Indice :* `balanced_accuracy_score(y_val, model.predict(X_val))` ; `max(val_scores, key=val_scores.get)`.

In [ ]:
K_LIST = [1, 3, 9, 27, 81]

In [ ]:
# --- your code
val_scores = None   # dict: k -> balanced accuracy on the validation set
best_k = None
# --- end
your_turn("K", globals())

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [ ]:
# Worked solution, open after you tried
# Names end in _sol so that running this cell keeps your own results.
val_scores_sol = {}
for k in K_LIST:
    model = make_knn(k).fit(X_train, y_train)
    val_scores_sol[k] = balanced_accuracy_score(y_val, model.predict(X_val))
best_k_sol = max(val_scores_sol, key=val_scores_sol.get)

In [ ]:
print("best k:", best_k, " validation balanced accuracy:", {k: round(v, 3) for k, v in val_scores.items()})
plot_k_curve(val_scores, best_k)

**À vous : testez une seule fois.** Ajustez à nouveau `make_knn(best_k)` sur l'ensemble d'entraînement, appelez-le `knn`, prédisez les événements de test 2022-2024 dans `y_pred_test`, et écrivez `recall_test`, le rappel de chaque classe sur l'ensemble de test (dans l'ordre de `CLASSES`).

*Indice :* `knn = make_knn(best_k).fit(X_train, y_train)` ; `recall_score(y_test, y_pred_test, labels=CLASSES, average=None)`.

In [ ]:
# --- your code
knn = None
y_pred_test = None
recall_test = None   # array: recall of earthquake, recall of explosion
# --- end
your_turn("T", globals())

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [ ]:
# Worked solution, open after you tried
# Names end in _sol so that running this cell keeps your own results.
knn_sol = make_knn(best_k).fit(X_train, y_train)
y_pred_test_sol = knn_sol.predict(X_test)
recall_test_sol = recall_score(y_test, y_pred_test_sol, labels=CLASSES, average=None)

In [ ]:
# Helper: Confusion matrices side by side, one per partition.
def plot_confusions(pairs):
    fig, ax = plt.subplots(1, len(pairs), figsize=(4.3 * len(pairs), 3.8))
    for a, (title, yt, yp) in zip(np.atleast_1d(ax), pairs):
        ConfusionMatrixDisplay(confusion_matrix(yt, yp, labels=CLASSES), display_labels=CLASSES).plot(
            ax=a, colorbar=False, cmap="Blues", values_format="d")
        a.set_title(title, fontsize=10)
        a.grid(False)
    fig.tight_layout()


In [ ]:
y_pred_val = knn.predict(X_val)
plot_confusions([("validation 2015-2021", y_val, y_pred_val), ("test 2022-2024", y_test, y_pred_test)])
for name, yt, yp in [("validation", y_val, y_pred_val), ("test", y_test, y_pred_test)]:
    r = recall_score(yt, yp, labels=CLASSES, average=None)
    print(f"{name:10s} recall earthquake {r[0]:.3f}  recall explosion {r[1]:.3f}  balanced accuracy {r.mean():.3f}")

Les lignes d'une matrice de confusion sont les étiquettes des analystes, les colonnes les prédictions ; les cases hors de la diagonale sont les erreurs. Le score de test est proche du score de validation : de 2015-2021 à 2022-2024, les sites miniers, les heures de travail et la façon dont les analystes du PNSN étiquettent et localisent les tirs ont peu changé, si bien que le modèle se transfère dans le temps. C'est un constat sur ce réseau et ces années. Il ne dit pas que le modèle fonctionnerait dans une autre région, où les carrières sont ailleurs.

## 3. Le raccourci : sur quoi le modèle s'appuie-t-il ?

Un classifieur qui s'appuie sur l'heure apprend l'horaire de travail des carrières. Il prendra un tir de nuit pour un séisme (des tirs de tunnel et de chantier ont bien lieu la nuit), et il doutera d'un séisme de jour, par exemple un événement d'un essaim qui se poursuit pendant la journée de travail. Le rapport jour/nuit de Wiemer et Baer est une statistique sur de nombreux événements ; un événement à 10:00 n'est pas un tir à cause de son heure. Le modèle a aussi le droit d'utiliser la profondeur, dont nous avons vu qu'elle est en partie fixée par l'analyste une fois l'étiquette décidée.

**À vous : retirez l'heure.** Ajustez à nouveau le classifieur k-NN (même `best_k`) sans la variable `hour` et prédisez l'ensemble de test dans `y_pred_nohour`.

*Indice :* `NOHOUR = [f for f in FEATURES if f != "hour"]`, puis ajustez sur `X_train[NOHOUR]` et prédisez `X_test[NOHOUR]`.

In [ ]:
# --- your code
y_pred_nohour = None
# --- end
your_turn("N", globals())

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [ ]:
# Worked solution, open after you tried
# Names end in _sol so that running this cell keeps your own results.
NOHOUR = [f for f in FEATURES if f != "hour"]
y_pred_nohour_sol = make_knn(best_k).fit(X_train[NOHOUR], y_train).predict(X_test[NOHOUR])

In [ ]:
# Helper: Recall on all test events, on blasts outside working hours and on earthquakes inside them, for several feature sets.
def shortcut_table(rows, te):
    """Recall on all test events, on blasts outside 08:00-17:00 and on earthquakes inside it."""
    night_ex = (te["label"] == "explosion") & ~te["hour"].between(8, 16)
    day_eq = (te["label"] == "earthquake") & te["hour"].between(8, 16)
    out = []
    for name, pred in rows.items():
        r = recall_score(te["label"], pred, labels=CLASSES, average=None)
        out.append({"features": name, "balanced accuracy": r.mean(), "recall earthquake": r[0],
                    "recall explosion": r[1],
                    f"blasts outside 08-17 h ({night_ex.sum()})": (pred[night_ex.to_numpy()] == "explosion").mean(),
                    f"earthquakes 08-17 h ({day_eq.sum()})": (pred[day_eq.to_numpy()] == "earthquake").mean()})
    return pd.DataFrame(out).set_index("features").round(3)


In [ ]:
LOCATION_ONLY = ["mag", "latitude", "longitude"]
y_pred_loc = make_knn(best_k).fit(X_train[LOCATION_ONLY], y_train).predict(X_test[LOCATION_ONLY])
te = cat.loc[~in_domain]
shortcut = shortcut_table({"all five": y_pred_test, "without hour": y_pred_nohour,
                           "without hour and depth": y_pred_loc}, te)
shortcut

Les colonnes de droite sont les événements qui contredisent le raccourci : tirs en dehors des heures de travail et séismes pendant ces heures. Avec l'heure, le modèle manque une partie des tirs hors des heures de travail ; sans elle, il en trouve davantage, et le score global bouge à peine parce que la profondeur et la localisation portent déjà l'information. Si l'on retire aussi la profondeur, le score baisse : il reste la localisation des mines, une variable légitime pour ce réseau, mais qui ne se transférerait pas à une nouvelle carrière. Les variables acceptables dépendent de l'usage prévu du modèle. S'il doit étiqueter les nouveaux événements avant qu'un analyste ne les voie, la profondeur fixée par l'analyste n'est pas disponible, et le score honnête est celui de la dernière ligne.

## 4. Régression : masse volumique d'une roche à partir de sa chimie en éléments majeurs

La masse volumique est une propriété globale d'une roche dont les modèles gravimétriques et sismiques ont besoin, et elle est mesurée sur bien moins d'échantillons que la chimie. Les roches mafiques sont riches en silicates de fer et de magnésium (olivine, pyroxène, amphibole) et sont plus denses que les roches felsiques, faites surtout de quartz et de feldspath. Peut-on prédire la masse volumique (g/cm³) à partir des sept oxydes d'éléments majeurs (wt%) ?

Nous utilisons la table synthétique d'analyses sur roche totale du cours (`mlgeo_synth.geochem_table`, comme en 2.4 et 2.12) : 5000 analyses de basalte, d'andésite et de granite. Ses masses volumiques suivent des moyennes par classe avec une faible tendance le long de la différenciation de chaque magma (la table est synthétique, on sait donc que la vraie relation est simple).

**Partition le long d'un axe physique.** L'entraînement et la validation utilisent les roches dont la teneur en SiO₂ est inférieure à 63 wt%, la limite entre roches intermédiaires et felsiques dans la classification total alcalins-silice (Le Bas et al., 1986). L'ensemble de test contient toutes les roches à 63 wt% ou plus : les roches les plus riches en silice, absentes de l'entraînement. C'est ce qui arrive quand un modèle calibré sur des basaltes et des andésites est appliqué à un pluton granitique.

In [ ]:
# Helper: Silica content and density of the rocks, with the threshold that separates training from the silica-rich test.
def plot_rocks(rocks, thr):
    fig, ax = plt.subplots(figsize=(7, 4))
    for lab in ["basalt", "andesite", "granite"]:
        m = rocks["label"] == lab
        ax.scatter(rocks.loc[m, "SIO2"], rocks.loc[m, "density_g_cm3"], s=4, alpha=0.4, color=COLORS[lab], label=lab)
    ax.axvline(thr, color="k", ls="--", lw=1)
    ax.text(thr - 0.5, 3.12, "train and validation  ", ha="right", fontsize=9)
    ax.text(thr + 0.5, 3.12, "out-of-domain test", ha="left", fontsize=9)
    ax.set(xlabel="SiO$_2$ (wt%)", ylabel="density (g/cm³)", title="Synthetic whole-rock analyses (mlgeo_synth)")
    ax.legend(fontsize=8, markerscale=4, loc="lower left")


In [ ]:
rocks = mlgeo_synth.geochem_table(n=5000, seed=42)
OXIDES = ["SIO2", "AL2O3", "FEO", "MGO", "CAO", "NA2O", "K2O"]
SIO2_MAX = 63.0
low = rocks["SIO2"] < SIO2_MAX
Xr_train, Xr_val, yr_train, yr_val = train_test_split(rocks.loc[low, OXIDES], rocks.loc[low, "density_g_cm3"],
                                                      test_size=0.25, random_state=42)
Xr_ood, yr_ood = rocks.loc[~low, OXIDES], rocks.loc[~low, "density_g_cm3"]
print(f"train {len(Xr_train)}, validation {len(Xr_val)}, silica-rich test {len(Xr_ood)}")
plot_rocks(rocks, SIO2_MAX)

**À vous : référence et régression linéaire.** Ajustez `base_r = DummyRegressor(strategy="mean")` et `lin = LinearRegression()` sur l'ensemble d'entraînement, et écrivez `val_mae`, un dictionnaire contenant l'erreur absolue moyenne (g/cm³) de chacun sur l'ensemble de validation, avec les clés `"mean"` et `"linear"`.

*Indice :* `mean_absolute_error(yr_val, lin.predict(Xr_val))`.

In [ ]:
# --- your code
base_r = None
lin = None
val_mae = None   # {"mean": ..., "linear": ...}, in g/cm3
# --- end
your_turn("R1", globals())

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [ ]:
# Worked solution, open after you tried
# Names end in _sol so that running this cell keeps your own results.
base_r_sol = DummyRegressor(strategy="mean").fit(Xr_train, yr_train)
lin_sol = LinearRegression().fit(Xr_train, yr_train)
val_mae_sol = {"mean": mean_absolute_error(yr_val, base_r_sol.predict(Xr_val)),
               "linear": mean_absolute_error(yr_val, lin_sol.predict(Xr_val))}

Deux modèles non linéaires : un polynôme de degré 2 (tous les produits de paires d'oxydes, puis un ajustement linéaire) et une forêt aléatoire. Nous choisissons parmi les quatre modèles d'après l'erreur absolue moyenne de validation, et donnons aussi le R².

In [ ]:
# Helper: Validation scores of several regression models in one table; used for the three candidate models.
def val_table_of(models, X, y):
    return pd.DataFrame({"MAE validation (g/cm3)": {n: mean_absolute_error(y, m.predict(X)) for n, m in models.items()},
                         "R2 validation": {n: r2_score(y, m.predict(X)) for n, m in models.items()}})


In [ ]:
models = {"mean": base_r, "linear": lin,
          "polynomial deg 2": make_pipeline(StandardScaler(), PolynomialFeatures(2), LinearRegression()).fit(Xr_train, yr_train),
          "random forest": RandomForestRegressor(200, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(Xr_train, yr_train)}
val_table = val_table_of(models, Xr_val, yr_val)
best_name = val_table["MAE validation (g/cm3)"].drop("mean").idxmin()
print("chosen on validation:", best_name)
val_table.round(4)

Les trois modèles sont à quelques millièmes de g/cm³ les uns des autres en validation, à moins de la moitié de l'erreur de la référence ; le bruit de mesure de la table synthétique est de 0,03 g/cm³, donc aucun ne peut faire beaucoup mieux. La validation ne permet pas de les départager.

**À vous : le test sur les roches riches en silice.** Écrivez `ood_mae`, l'erreur absolue moyenne de chaque modèle de `models` sur l'ensemble de test riche en silice (un dictionnaire avec les mêmes clés que `models`), et `frac_physical`, la fraction des prédictions de test du modèle retenu comprises entre 2,5 et 3,3 g/cm³ (la gamme des roches ignées courantes de la croûte).

*Indice :* une compréhension de dictionnaire sur `models.items()` ; `((p >= 2.5) & (p <= 3.3)).mean()`.

In [ ]:
# --- your code
ood_mae = None         # dict: model name -> MAE (g/cm3) on rocks with SiO2 >= 63 wt%
frac_physical = None   # fraction of models[best_name] predictions within 2.5-3.3 g/cm3
# --- end
your_turn("R2", globals())

*Solution détaillée ci-dessous (cliquez sur ⋯ pour l'ouvrir).*

In [ ]:
# Worked solution, open after you tried
# Names end in _sol so that running this cell keeps your own results.
ood_mae_sol = {name: mean_absolute_error(yr_ood, m.predict(Xr_ood)) for name, m in models.items()}
p_sol = models[best_name].predict(Xr_ood)
frac_physical_sol = ((p_sol >= 2.5) & (p_sol <= 3.3)).mean()

In [ ]:
# Helper: Predicted against observed density within and outside the training range, and the same models refit on a random split.
def plot_pred_obs(models, Xv, yv, Xo, yo):
    fig, ax = plt.subplots(1, len(models), figsize=(4 * len(models), 3.8), sharex=True, sharey=True)
    for a, (name, m) in zip(ax, models.items()):
        a.plot(yv, m.predict(Xv), ".", ms=2, alpha=0.4, color="0.4", label="validation (SiO$_2$ < threshold)")
        a.plot(yo, m.predict(Xo), ".", ms=2, alpha=0.4, color="C0", label="test (silica-rich)")
        a.plot([2.5, 3.2], [2.5, 3.2], "k-", lw=0.8)
        a.set(title=name, xlabel="observed density (g/cm³)")
    ax[0].set_ylabel("predicted density (g/cm³)")
    ax[0].legend(fontsize=7, markerscale=5, loc="upper left")
    fig.tight_layout()


def random_split_mae(rocks, models, oxides):
    """Same models refit on a random 80/20 split of all rocks: MAE on the random 20%."""
    Xa, Xb, ya, yb = train_test_split(rocks[oxides], rocks["density_g_cm3"], test_size=0.2, random_state=42)
    return {n: mean_absolute_error(yb, clone(m).fit(Xa, ya).predict(Xb)) for n, m in models.items()}


In [ ]:
reg = pd.DataFrame({"validation (SiO2 < 63)": val_table["MAE validation (g/cm3)"], "silica-rich test": ood_mae,
                    "random 20% test": random_split_mae(rocks, models, OXIDES)}).round(4)
print(f"chosen model ({best_name}): {100 * frac_physical:.1f}% of test predictions within 2.5-3.3 g/cm3")
plot_pred_obs({n: m for n, m in models.items() if n != "mean"}, Xr_val, yr_val, Xr_ood, yr_ood)
reg

Toutes les prédictions sont physiquement plausibles, et pourtant les erreurs augmentent sur les roches riches en silice. Le modèle linéaire extrapole la tendance du basalte à l'andésite et tombe près des masses volumiques du granite ; son erreur augmente d'environ un cinquième. La forêt aléatoire ne peut pas extrapoler : chaque prédiction est une moyenne de masses volumiques d'entraînement, si bien qu'un granite reçoit la masse volumique des andésites les plus évoluées, environ 0,1 g/cm³ de trop, à peu près quatre fois son erreur de validation. Une partition aléatoire cache le problème, car chaque roche de test a alors des voisines du même type dans l'ensemble d'entraînement. Deux modèles à égalité en validation diffèrent d'un facteur supérieur à trois hors du domaine, et seul un test hors domaine le montre.

## 5. Résumé

In [ ]:
# Helper: The summary table at the end of the notebook.
def summarize(val_bal, test_bal, val_mae_base, val_mae_best, ood_mae_best):
    return pd.DataFrame([
        {"task": "classification", "target": "analyst label, earthquake or explosion", "metric": "balanced accuracy",
         "baseline": 0.5, "best model, validation": val_bal, "best model, out of domain": test_bal,
         "out-of-domain axis": "time: 2022-2024"},
        {"task": "regression", "target": "density (g/cm3)", "metric": "MAE (g/cm3)", "baseline": val_mae_base,
         "best model, validation": val_mae_best, "best model, out of domain": ood_mae_best,
         "out-of-domain axis": "composition: SiO2 >= 63 wt%"}]).set_index("task").round(3)


In [ ]:
summarize(val_scores[best_k], recall_test.mean(), val_mae["mean"], val_table.loc[best_name, "MAE validation (g/cm3)"],
          ood_mae[best_name])

Les deux tâches utilisaient des étiquettes ou des cibles mesurées. La séance 13 et le notebook [3.3](3.3_clustering.ipynb) retirent les étiquettes : le *clustering* (partitionnement) cherche la même structure (groupes d'événements, familles de roches) à partir des seules variables.

Pour le critère B7 de la grille d'évaluation des données prêtes pour l'IA en [2.13](../Chapter2-DataManipulation/2.13_MLready_data.ipynb), indiquez l'axe de votre ensemble de test (temps, espace, composition) et donnez côte à côte les scores de validation et de test, comme dans le tableau ci-dessus.